In [92]:
import requests
import pandas as pd
import json

In [93]:
url = "https://api.gdc.cancer.gov/files"

response = requests.post(
    url,
    headers={"Content-Type": "application/json"}
)

print(response.status_code)

200


In [94]:
filters = {
    "op": "and",
    "content": [
        {
            "op": "=",
            "content": {
                "field": "cases.project.project_id",
                "value": "TCGA-BRCA"
            }
        },
        {
            "op": "=",
            "content": {
                "field": "access",
                "value": "open"
            }
        }
    ]
}

response = requests.post(
    url,
    headers={"Content-Type": "application/json"},
    json={"filters": filters}
)

print(response.status_code)

200


In [7]:
data = response.json()

print(type(data))
print(data.keys())

<class 'dict'>
dict_keys(['data', 'warnings'])


In [8]:
data["data"].keys()

dict_keys(['hits', 'pagination'])

In [9]:
data["data"]["pagination"]

{'count': 10,
 'total': 27931,
 'size': 10,
 'from': 0,
 'sort': '',
 'page': 1,
 'pages': 2794}

In [10]:
data["data"]["hits"][0]

{'id': '757ee75b-674e-49a5-8ba0-cc1b574a51ae',
 'data_format': 'TXT',
 'access': 'open',
 'file_name': '31fa9bfd-dc91-4e27-be93-db7df8df2cd8.mirbase21.mirnas.quantification.txt',
 'submitter_id': 'mirna_swap_dr11_1186_MirnaExpression43d41344-73b1-4786-8b53-93e420b61c0b_profiling',
 'data_category': 'Transcriptome Profiling',
 'acl': ['open'],
 'type': 'mirna_expression',
 'platform': 'Illumina',
 'file_size': 50084,
 'created_datetime': '2018-03-20T04:53:45.847187-05:00',
 'md5sum': '8f2fb8583ec2ea249dd11fc3fa5ce78c',
 'updated_datetime': '2024-07-29T18:10:34.696357-05:00',
 'file_id': '757ee75b-674e-49a5-8ba0-cc1b574a51ae',
 'data_type': 'miRNA Expression Quantification',
 'state': 'released',
 'experimental_strategy': 'miRNA-Seq',
 'version': '1',
 'data_release': '12.0 - 46.0'}

In [11]:
fields = [
    "file_id",
    "file_name",
    "data_type",
    "access",
    "cases.case_id",
    "cases.submitter_id"
]

In [14]:
response = requests.post(
    url,
    headers={"Content-Type": "application/json"},
    json={
        "filters": filters,
        "fields": ",".join(fields),
        "size": 100
    }
)

In [15]:
data = response.json()
data["data"]["hits"][0]

{'id': '757ee75b-674e-49a5-8ba0-cc1b574a51ae',
 'cases': [{'case_id': 'dfefb76a-ec6b-4cd2-9d45-2e1e4befc7ea',
   'submitter_id': 'TCGA-A8-A09E'}],
 'access': 'open',
 'file_name': '31fa9bfd-dc91-4e27-be93-db7df8df2cd8.mirbase21.mirnas.quantification.txt',
 'file_id': '757ee75b-674e-49a5-8ba0-cc1b574a51ae',
 'data_type': 'miRNA Expression Quantification'}

In [16]:
hits = data["data"]["hits"]

print(len(hits))

100


In [17]:
df = pd.json_normalize(hits)
df.head()

,id,cases,access,file_name,file_id,data_type
0,757ee75b-674e-49a5-8ba0-cc1b574a51ae,[{'case_id': 'dfefb76a-ec6b-4cd2-9d45-2e1e4bef...,open,31fa9bfd-dc91-4e27-be93-db7df8df2cd8.mirbase21...,757ee75b-674e-49a5-8ba0-cc1b574a51ae,miRNA Expression Quantification
1,28a744f9-3ce6-41c9-9cd1-11b8f85a6f3e,[{'case_id': 'dfefb76a-ec6b-4cd2-9d45-2e1e4bef...,open,TCGA-BRCA.70b9ece6-dedc-44c7-8f35-b78d9bac626e...,28a744f9-3ce6-41c9-9cd1-11b8f85a6f3e,Gene Level Copy Number
2,7dc5a6ac-d1e1-416b-87dd-1a99a522ce15,[{'case_id': 'dfefb76a-ec6b-4cd2-9d45-2e1e4bef...,open,TCGA-A8-A09E-01A-21-A13A-20_RPPA_data.tsv,7dc5a6ac-d1e1-416b-87dd-1a99a522ce15,Protein Expression Quantification
3,b6e8df85-fcd5-47e3-b2aa-13894cbc4326,[{'case_id': 'dfefb76a-ec6b-4cd2-9d45-2e1e4bef...,open,nationwidechildrens.org_omf.TCGA-A8-A09E.xml,b6e8df85-fcd5-47e3-b2aa-13894cbc4326,Clinical Supplement
4,172eaec0-f64d-40e2-846a-cf4bc119b779,[{'case_id': 'dfefb76a-ec6b-4cd2-9d45-2e1e4bef...,open,60ea9ace-996e-40d4-98a7-1f05a1df00de_noid_Grn....,172eaec0-f64d-40e2-846a-cf4bc119b779,Masked Intensities


In [20]:
print(df.shape)

(100, 6)


In [21]:
print(df.columns.tolist())

['id', 'cases', 'access', 'file_name', 'file_id', 'data_type']


In [22]:
df["data_type"].value_counts()

data_type
Slide Image                            16
Masked Intensities                     12
Copy Number Segment                    12
Gene Level Copy Number                 10
Masked Copy Number Segment              8
Allele-specific Copy Number Segment     7
miRNA Expression Quantification         5
Biospecimen Supplement                  5
Gene Expression Quantification          5
Isoform Expression Quantification       4
Methylation Beta Value                  4
Protein Expression Quantification       3
Clinical Supplement                     3
Masked Somatic Mutation                 3
Pathology Report                        3
Name: count, dtype: int64

# WSI Filter

In [23]:
wsi_filters = {
    "op": "and",
    "content": [
        {
            "op": "=",
            "content": {
                "field": "cases.project.project_id",
                "value": "TCGA-BRCA"
            }
        },
        {
            "op": "=",
            "content": {
                "field": "access",
                "value": "open"
            }
        },
        {
            "op": "=",
            "content": {
                "field": "data_type",
                "value": "Slide Image"
            }
        }
    ]
}

In [24]:
wsi_response = requests.post(
    url,
    headers={"Content-Type": "application/json"},
    json={
        "filters": wsi_filters,
        "fields": "file_id,file_name,data_type,access,cases.case_id,cases.submitter_id",
        "size": 1000
    }
)

print(wsi_response.status_code)

200


In [25]:
wsi_data = wsi_response.json()

wsi_data["data"]["pagination"]

{'count': 1000,
 'total': 3112,
 'size': 1000,
 'from': 0,
 'sort': '',
 'page': 1,
 'pages': 4}

In [26]:
wsi_data["data"]["pagination"]["total"]

3112

In [29]:
all_wsi_hits = []

for page in range(1, 5):
    response = requests.post(
        url,
        headers={"Content-Type": "application/json"},
        json={
            "filters": wsi_filters,
            "fields": "file_id,file_name,data_type,access,cases.case_id,cases.submitter_id",
            "size": 1000,
            "from": (page - 1) * 1000
        }
    )

    page_data = response.json()["data"]["hits"]
    all_wsi_hits.extend(page_data)

print(len(all_wsi_hits))

3112


In [30]:
wsi_cases = {
    case["submitter_id"]
    for hit in all_wsi_hits
    for case in hit.get("cases", [])
}

print(len(wsi_cases))

1098


# RNA Filter

In [31]:
rna_filters = {
    "op": "and",
    "content": [
        {
            "op": "=",
            "content": {
                "field": "cases.project.project_id",
                "value": "TCGA-BRCA"
            }
        },
        {
            "op": "=",
            "content": {
                "field": "access",
                "value": "open"
            }
        },
        {
            "op": "=",
            "content": {
                "field": "data_type",
                "value": "Gene Expression Quantification"
            }
        }
    ]
}

In [33]:
rna_response = requests.post(
    url,
    headers={"Content-Type": "application/json"},
    json={
        "filters": rna_filters,
        "fields": "file_id,file_name,data_type,access,cases.case_id,cases.submitter_id",
        "size": 1000
    }
)

print(rna_response.status_code)

200


In [34]:
rna_response.json()["data"]["pagination"]

{'count': 1000,
 'total': 1231,
 'size': 1000,
 'from': 0,
 'sort': '',
 'page': 1,
 'pages': 2}

In [36]:
all_rna_hits = []

for page in range(1, 3):
    response = requests.post(
        url,
        headers={"Content-Type": "application/json"},
        json={
            "filters": rna_filters,
            "fields": "file_id,file_name,data_type,access,cases.case_id,cases.submitter_id",
            "size": 1000,
            "from": (page - 1) * 1000
        }
    )

    page_data = response.json()["data"]["hits"]
    all_rna_hits.extend(page_data)

print(len(all_rna_hits))

1231


In [37]:
rna_cases = {
    case["submitter_id"]
    for hit in all_rna_hits
    for case in hit.get("cases", [])
}

print(len(rna_cases))

1095


In [38]:
common_cases = wsi_cases & rna_cases

print(len(common_cases))

1095


In [39]:
wsi_only = wsi_cases - rna_cases

print(len(wsi_only))
print(wsi_only)

3
{'TCGA-AR-A0U1', 'TCGA-AC-A5EI', 'TCGA-C8-A9FZ'}


In [40]:
rna_only = rna_cases - wsi_cases

print(len(rna_only))
print(rna_only)

0
set()


In [41]:
example_case = next(iter(common_cases))

print(example_case)

TCGA-LL-A7T0


In [42]:
example_wsi = [
    hit for hit in all_wsi_hits
    if any(case["submitter_id"] == example_case
           for case in hit.get("cases", []))
]

example_rna = [
    hit for hit in all_rna_hits
    if any(case["submitter_id"] == example_case
           for case in hit.get("cases", []))
]

print("WSI files:", len(example_wsi))
print("RNA files:", len(example_rna))

WSI files: 2
RNA files: 1


In [43]:
pd.DataFrame(example_wsi)[
    ["file_id", "file_name", "data_type", "access"]
]

,file_id,file_name,data_type,access
0,a8ec7eb3-4e63-4368-b45d-57c2cdfe1f45,TCGA-LL-A7T0-01Z-00-DX1.B03BBA63-ACF4-4BCA-9F2...,Slide Image,open
1,30750db4-9ff0-4236-b0a8-04229b70c1e0,TCGA-LL-A7T0-01A-03-TS3.07761D74-57D6-4859-AD0...,Slide Image,open


In [44]:
pd.DataFrame(example_rna)[
    ["file_id", "file_name", "data_type", "access"]
]

,file_id,file_name,data_type,access
0,515f8866-8744-44d8-9e4b-0e734d0c201b,c5d73777-b7f3-4107-9efc-afde92f70469.rna_seq.a...,Gene Expression Quantification,open


In [45]:
print(example_rna[0]["file_name"])

c5d73777-b7f3-4107-9efc-afde92f70469.rna_seq.augmented_star_gene_counts.tsv


In [46]:
for hit in example_wsi:
    print(hit["file_name"])

TCGA-LL-A7T0-01Z-00-DX1.B03BBA63-ACF4-4BCA-9F2B-F631F0C6A25C.svs
TCGA-LL-A7T0-01A-03-TS3.07761D74-57D6-4859-AD0B-BD2F8C68D905.svs


In [47]:
case_url = "https://api.gdc.cancer.gov/cases"

case_response = requests.post(
    case_url,
    headers={"Content-Type": "application/json"},
    json={
        "filters": {
            "op": "=",
            "content": {
                "field": "submitter_id",
                "value": example_case
            }
        },
        "format": "JSON"
    }
)

print(case_response.status_code)

200


In [48]:
case_data = case_response.json()

case_data.keys()

dict_keys(['data', 'warnings'])

In [49]:
case_data["data"]["hits"][0].keys()

dict_keys(['id', 'lost_to_followup', 'slide_ids', 'submitter_slide_ids', 'disease_type', 'analyte_ids', 'submitter_id', 'submitter_analyte_ids', 'days_to_consent', 'aliquot_ids', 'submitter_aliquot_ids', 'created_datetime', 'diagnosis_ids', 'sample_ids', 'consent_type', 'submitter_sample_ids', 'primary_site', 'submitter_diagnosis_ids', 'updated_datetime', 'case_id', 'index_date', 'state', 'portion_ids', 'submitter_portion_ids'])

In [50]:
case = case_data["data"]["hits"][0]

print("Case:", case["submitter_id"])
print("Samples:", case["submitter_sample_ids"])
print("Slides:", case["submitter_slide_ids"])
print("Analytes:", case["submitter_analyte_ids"])

Case: TCGA-LL-A7T0
Samples: ['TCGA-LL-A7T0-10A', 'TCGA-LL-A7T0-01A', 'TCGA-LL-A7T0-01Z']
Slides: ['TCGA-LL-A7T0-01A-03-TS3', 'TCGA-LL-A7T0-01Z-00-DX1']
Analytes: ['TCGA-LL-A7T0-01A-31D', 'TCGA-LL-A7T0-10A-01D', 'TCGA-LL-A7T0-01A-31R', 'TCGA-LL-A7T0-01A-41D', 'TCGA-LL-A7T0-10A-01W', 'TCGA-LL-A7T0-01A-31W']


In [51]:
print("Disease:", case["disease_type"])
print("Primary site:", case["primary_site"])

Disease: Ductal and Lobular Neoplasms
Primary site: Breast


In [53]:
print(case_data["data"]["hits"][0]["diagnosis_ids"])

['87c779e4-f86f-573d-b584-97740dbc24fc']


In [54]:
for key, value in case.items():
    print(f"{key}: {value}")

id: d8f8064f-02ef-4fed-942b-714cbe5e8455
lost_to_followup: No
slide_ids: ['0a94decd-c84d-4b79-ab5f-65b3ab54c571', '07761d74-57d6-4859-ad0b-bd2f8c68d905']
submitter_slide_ids: ['TCGA-LL-A7T0-01A-03-TS3', 'TCGA-LL-A7T0-01Z-00-DX1']
disease_type: Ductal and Lobular Neoplasms
analyte_ids: ['aaf496a5-4f48-4ac5-b1e3-245ae0778394', '74c2a2dc-35a5-4ff8-b5f2-f43656198754', '6e3c2de1-1e67-435a-a805-78e7e5108974', '94837eb6-8d86-497d-9845-f9f5f967b996', 'e3aec9b3-5e34-4a8c-abab-95ff2ee902e4', '8e2d33f1-ceb0-48fc-89d7-e62574733302']
submitter_id: TCGA-LL-A7T0
submitter_analyte_ids: ['TCGA-LL-A7T0-01A-31D', 'TCGA-LL-A7T0-10A-01D', 'TCGA-LL-A7T0-01A-31R', 'TCGA-LL-A7T0-01A-41D', 'TCGA-LL-A7T0-10A-01W', 'TCGA-LL-A7T0-01A-31W']
days_to_consent: 50
aliquot_ids: ['67e967d6-be17-4fe1-a1fb-197655fad18d', '0c85852a-3bc0-40a1-be1a-ce9e8a90e91f', '13199461-7dd2-4ae0-a478-5afbb61f8578', '47f88f2d-846a-4281-b7f5-3888768ae4e9', 'db0b8004-7649-4009-8974-4f34785f3cf3', 'a6f61b58-a3d5-46f0-8032-058559ecba10', 'b8d

In [55]:
label_fields = [
    "cases.submitter_id",
    "cases.diagnoses.primary_diagnosis",
    "cases.diagnoses.morphology",
    "cases.diagnoses.tumor_grade",
    "cases.diagnoses.tumor_stage",
    "cases.diagnoses.ajcc_pathologic_stage",
    "cases.diagnoses.vital_status",
    "cases.diagnoses.last_known_disease_status",
    "cases.diagnoses.progression_or_recurrence",
]

label_response = requests.post(
    case_url,
    headers={"Content-Type": "application/json"},
    json={
        "filters": {
            "op": "=",
            "content": {
                "field": "submitter_id",
                "value": example_case
            }
        },
        "fields": ",".join(label_fields),
        "format": "JSON"
    }
)

print(label_response.status_code)

200


In [56]:
label_data = label_response.json()

label_data["data"]["hits"][0]

{'id': 'd8f8064f-02ef-4fed-942b-714cbe5e8455',
 'lost_to_followup': 'No',
 'slide_ids': ['0a94decd-c84d-4b79-ab5f-65b3ab54c571',
  '07761d74-57d6-4859-ad0b-bd2f8c68d905'],
 'submitter_slide_ids': ['TCGA-LL-A7T0-01A-03-TS3', 'TCGA-LL-A7T0-01Z-00-DX1'],
 'disease_type': 'Ductal and Lobular Neoplasms',
 'analyte_ids': ['aaf496a5-4f48-4ac5-b1e3-245ae0778394',
  '74c2a2dc-35a5-4ff8-b5f2-f43656198754',
  '6e3c2de1-1e67-435a-a805-78e7e5108974',
  '94837eb6-8d86-497d-9845-f9f5f967b996',
  'e3aec9b3-5e34-4a8c-abab-95ff2ee902e4',
  '8e2d33f1-ceb0-48fc-89d7-e62574733302'],
 'submitter_id': 'TCGA-LL-A7T0',
 'submitter_analyte_ids': ['TCGA-LL-A7T0-01A-31D',
  'TCGA-LL-A7T0-10A-01D',
  'TCGA-LL-A7T0-01A-31R',
  'TCGA-LL-A7T0-01A-41D',
  'TCGA-LL-A7T0-10A-01W',
  'TCGA-LL-A7T0-01A-31W'],
 'days_to_consent': 50,
 'aliquot_ids': ['67e967d6-be17-4fe1-a1fb-197655fad18d',
  '0c85852a-3bc0-40a1-be1a-ce9e8a90e91f',
  '13199461-7dd2-4ae0-a478-5afbb61f8578',
  '47f88f2d-846a-4281-b7f5-3888768ae4e9',
  'db0b80

In [59]:
test_fields = [
    "submitter_id",
    "diagnoses"
]

test_response = requests.post(
    case_url,
    headers={"Content-Type": "application/json"},
    json={
        "filters": {
            "op": "=",
            "content": {
                "field": "submitter_id",
                "value": example_case
            }
        },
        "fields": ",".join(test_fields),
        "format": "JSON"
    }
)

print(test_response.status_code)
print(test_response.json()["data"]["hits"][0])

200
{'id': 'd8f8064f-02ef-4fed-942b-714cbe5e8455', 'submitter_id': 'TCGA-LL-A7T0'}


In [60]:
test_response = requests.post(
    case_url,
    headers={"Content-Type": "application/json"},
    json={
        "filters": {
            "op": "=",
            "content": {
                "field": "submitter_id",
                "value": example_case
            }
        },
        "fields": "submitter_id,diagnoses.primary_diagnosis,diagnoses.tumor_grade,diagnoses.tumor_stage",
        "format": "JSON"
    }
)

print(test_response.status_code)
print(test_response.json()["data"]["hits"][0])

200
{'id': 'd8f8064f-02ef-4fed-942b-714cbe5e8455', 'submitter_id': 'TCGA-LL-A7T0', 'diagnoses': [{'primary_diagnosis': 'Infiltrating duct carcinoma, NOS', 'tumor_grade': None}]}


In [63]:
label_response = requests.post(
    case_url,
    headers={"Content-Type": "application/json"},
    json={
        "filters": {
            "op": "in",
            "content": {
                "field": "submitter_id",
                "value": list(common_cases)
            }
        },
        "fields": (
            "submitter_id,"
            "diagnoses.primary_diagnosis,"
            "diagnoses.tumor_grade,"
            "diagnoses.tumor_stage,"
            "diagnoses.ajcc_pathologic_stage,"
            "diagnoses.vital_status,"
            "diagnoses.last_known_disease_status,"
            "diagnoses.progression_or_recurrence"
        ),
        "format": "JSON",
        "size": 2000
    }
)

print(label_response.status_code)

label_hits = label_response.json()["data"]["hits"]

print("Cases returned:", len(label_hits))
print(label_hits[0])

200
Cases returned: 1095
{'id': '57a1604c-60b7-4b30-a75e-f70939532c5c', 'submitter_id': 'TCGA-BH-A0B2'}


In [67]:
small_cases = list(common_cases)[:10]

test_response = requests.post(
    case_url,
    headers={"Content-Type": "application/json"},
    json={
        "filters": {
            "op": "in",
            "content": {
                "field": "submitter_id",
                "value": small_cases
            }
        },
        "fields": (
            "submitter_id,"
            "diagnoses.primary_diagnosis,"
            "diagnoses.tumor_grade"
        ),
        "format": "JSON",
        "size": 20
    }
)

print(test_response.status_code)

small_hits = test_response.json()["data"]["hits"]

print("Cases:", len(small_hits))
print(small_hits[0])

200
Cases: 10
{'id': 'c0892598-1f7b-4f23-9cd8-731f797753d5', 'submitter_id': 'TCGA-A2-A0YG', 'diagnoses': [{'primary_diagnosis': 'Infiltrating duct carcinoma, NOS', 'tumor_grade': None}]}


In [68]:
batch_size = 100
all_label_hits = []

common_cases_list = list(common_cases)

for start in range(0, len(common_cases_list), batch_size):

    batch = common_cases_list[start:start + batch_size]

    response = requests.post(
        case_url,
        headers={"Content-Type": "application/json"},
        json={
            "filters": {
                "op": "in",
                "content": {
                    "field": "submitter_id",
                    "value": batch
                }
            },
            "fields": (
                "submitter_id,"
                "diagnoses.primary_diagnosis,"
                "diagnoses.tumor_grade,"
                "diagnoses.tumor_stage,"
                "diagnoses.ajcc_pathologic_stage,"
                "diagnoses.vital_status,"
                "diagnoses.last_known_disease_status,"
                "diagnoses.progression_or_recurrence"
            ),
            "format": "JSON",
            "size": batch_size
        }
    )

    response.raise_for_status()

    hits = response.json()["data"]["hits"]
    all_label_hits.extend(hits)

    print(
        f"Retrieved {len(all_label_hits)} / {len(common_cases_list)} cases"
    )

Retrieved 100 / 1095 cases
Retrieved 200 / 1095 cases
Retrieved 300 / 1095 cases
Retrieved 400 / 1095 cases
Retrieved 500 / 1095 cases
Retrieved 600 / 1095 cases
Retrieved 700 / 1095 cases
Retrieved 800 / 1095 cases
Retrieved 900 / 1095 cases
Retrieved 1000 / 1095 cases
Retrieved 1095 / 1095 cases


In [69]:
print("Cases:", len(all_label_hits))

print("Example:")
print(all_label_hits[0])

Cases: 1095
Example:
{'id': '14b95463-2108-4921-afc2-e29eef52b18f', 'submitter_id': 'TCGA-A2-A0YT', 'diagnoses': [{'primary_diagnosis': 'Infiltrating duct carcinoma, NOS', 'ajcc_pathologic_stage': 'Stage IIIB', 'tumor_grade': None, 'progression_or_recurrence': None, 'last_known_disease_status': None}]}


In [70]:
rows = []

for case in all_label_hits:
    diagnosis = case.get("diagnoses", [{}])[0]

    rows.append({
        "case_id": case["submitter_id"],
        "primary_diagnosis": diagnosis.get("primary_diagnosis"),
        "tumor_grade": diagnosis.get("tumor_grade"),
        "tumor_stage": diagnosis.get("tumor_stage"),
        "ajcc_pathologic_stage": diagnosis.get("ajcc_pathologic_stage"),
        "vital_status": diagnosis.get("vital_status"),
        "last_known_disease_status": diagnosis.get("last_known_disease_status"),
        "progression_or_recurrence": diagnosis.get("progression_or_recurrence"),
    })

labels_df = pd.DataFrame(rows)

labels_df.head()

,case_id,primary_diagnosis,tumor_grade,tumor_stage,ajcc_pathologic_stage,vital_status,last_known_disease_status,progression_or_recurrence
0,TCGA-A2-A0YT,"Infiltrating duct carcinoma, NOS",NaN,None,Stage IIIB,None,NaN,NaN
1,TCGA-A8-A09E,"Infiltrating duct carcinoma, NOS",NaN,None,Stage IIIB,None,NaN,NaN
2,TCGA-AO-A0JI,"Infiltrating duct carcinoma, NOS",NaN,None,Stage IIA,None,NaN,NaN
3,TCGA-BH-A0C0,"Infiltrating duct carcinoma, NOS",NaN,None,Stage IIA,None,NaN,NaN
4,TCGA-AO-A1KQ,"Infiltrating duct carcinoma, NOS",NaN,None,Stage IIIB,None,NaN,NaN


In [71]:
labels_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1095 entries, 0 to 1094
Data columns (total 8 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   case_id                    1095 non-null   str   
 1   primary_diagnosis          1094 non-null   str   
 2   tumor_grade                1 non-null      str   
 3   tumor_stage                0 non-null      object
 4   ajcc_pathologic_stage      994 non-null    str   
 5   vital_status               0 non-null      object
 6   last_known_disease_status  1 non-null      str   
 7   progression_or_recurrence  1 non-null      str   
dtypes: object(2), str(6)
memory usage: 68.6+ KB


In [72]:
labels_df["ajcc_pathologic_stage"].value_counts()

ajcc_pathologic_stage
Stage IIA     334
Stage IIB     233
Stage IIIA    138
Stage IA       85
Stage I        83
Stage IIIC     56
Stage IIIB     23
Stage IV       18
Stage II        7
Stage X         7
Stage IB        6
Stage 0         4
Name: count, dtype: int64

In [73]:
labels_df["primary_diagnosis"].value_counts()

primary_diagnosis
Infiltrating duct carcinoma, NOS                            751
Lobular carcinoma, NOS                                      186
Not Reported                                                 34
Infiltrating duct and lobular carcinoma                      28
Infiltrating duct mixed with other types of carcinoma        18
Mucinous adenocarcinoma                                      15
Metaplastic carcinoma, NOS                                   13
Intraductal papillary adenocarcinoma with invasion            6
Infiltrating lobular mixed with other types of carcinoma      5
Medullary carcinoma, NOS                                      5
Invasive micropapillary carcinoma                             4
Intraductal carcinoma, noninfiltrating, NOS                   4
Paget disease and infiltrating duct carcinoma of breast       3
Pleomorphic carcinoma                                         3
Lobular carcinoma in situ, NOS                                3
Basal cell carcinoma, 

In [74]:
labeled_cases = set(
    labels_df.loc[
        labels_df["ajcc_pathologic_stage"].notna(),
        "case_id"
    ]
)

print("Labeled cases:", len(labeled_cases))

labeled_wsi = [
    hit for hit in all_wsi_hits
    if any(
        case["submitter_id"] in labeled_cases
        for case in hit.get("cases", [])
    )
]

labeled_rna = [
    hit for hit in all_rna_hits
    if any(
        case["submitter_id"] in labeled_cases
        for case in hit.get("cases", [])
    )
]

print("WSI files:", len(labeled_wsi))
print("RNA files:", len(labeled_rna))

Labeled cases: 994
WSI files: 2821
RNA files: 1120


In [75]:
print("Unique WSI cases:", len({
    case["submitter_id"]
    for hit in labeled_wsi
    for case in hit.get("cases", [])
}))

print("Unique RNA cases:", len({
    case["submitter_id"]
    for hit in labeled_rna
    for case in hit.get("cases", [])
}))

Unique WSI cases: 994
Unique RNA cases: 994


In [76]:
wsi_per_case = (
    pd.Series([
        case["submitter_id"]
        for hit in labeled_wsi
        for case in hit.get("cases", [])
    ])
    .value_counts()
)

print(wsi_per_case.describe())
print()
print(wsi_per_case.value_counts().sort_index())

count    994.000000
mean       2.838028
std        1.146351
min        1.000000
25%        2.000000
50%        3.000000
75%        3.000000
max        9.000000
Name: count, dtype: float64

count
1     17
2    432
3    404
4     46
5     48
6     32
7      9
8      5
9      1
Name: count, dtype: int64


In [77]:
example_rna_file = all_rna_hits[0]

example_rna_file

{'id': 'ead53b27-6ad9-4b96-b5d4-0d4f06fb2d13',
 'cases': [{'case_id': 'dfefb76a-ec6b-4cd2-9d45-2e1e4befc7ea',
   'submitter_id': 'TCGA-A8-A09E'}],
 'access': 'open',
 'file_name': 'dadd1789-354a-4abb-8d4e-8556166cf341.rna_seq.augmented_star_gene_counts.tsv',
 'file_id': 'ead53b27-6ad9-4b96-b5d4-0d4f06fb2d13',
 'data_type': 'Gene Expression Quantification'}

In [78]:
example_rna_file["file_id"]

'ead53b27-6ad9-4b96-b5d4-0d4f06fb2d13'

In [79]:
example_rna_file["file_name"]

'dadd1789-354a-4abb-8d4e-8556166cf341.rna_seq.augmented_star_gene_counts.tsv'

In [80]:
file_id = example_rna_file["file_id"]

data_url = f"https://api.gdc.cancer.gov/data/{file_id}"

response = requests.get(data_url)

print(response.status_code)
print(len(response.content))

200
4235051


In [81]:
with open("example_rna.tsv", "wb") as f:
    f.write(response.content)

In [85]:
rna_df = pd.read_csv(
    "example_rna.tsv",
    sep="\t",
    skiprows=1
)

rna_df.head()

,gene_id,gene_name,gene_type,unstranded,stranded_first,stranded_second,tpm_unstranded,fpkm_unstranded,fpkm_uq_unstranded
0,N_unmapped,NaN,NaN,2976444,2976444,2976444,NaN,NaN,NaN
1,N_multimapping,NaN,NaN,4105418,4105418,4105418,NaN,NaN,NaN
2,N_noFeature,NaN,NaN,1954586,28629612,29044609,NaN,NaN,NaN
3,N_ambiguous,NaN,NaN,5750570,1468817,1446495,NaN,NaN,NaN
4,ENSG00000000003.15,TSPAN6,protein_coding,3928,2013,1915,59.4987,17.2434,16.5964


In [86]:
rna_df.shape

(60664, 9)

In [84]:
with open("example_rna.tsv", "r") as f:
    for _ in range(10):
        print(repr(f.readline()))

'# gene-model: GENCODE v36\n'
'gene_id\tgene_name\tgene_type\tunstranded\tstranded_first\tstranded_second\ttpm_unstranded\tfpkm_unstranded\tfpkm_uq_unstranded\n'
'N_unmapped\t\t\t2976444\t2976444\t2976444\t\t\t\n'
'N_multimapping\t\t\t4105418\t4105418\t4105418\t\t\t\n'
'N_noFeature\t\t\t1954586\t28629612\t29044609\t\t\t\n'
'N_ambiguous\t\t\t5750570\t1468817\t1446495\t\t\t\n'
'ENSG00000000003.15\tTSPAN6\tprotein_coding\t3928\t2013\t1915\t59.4987\t17.2434\t16.5964\n'
'ENSG00000000005.6\tTNMD\tprotein_coding\t7\t2\t5\t0.3259\t0.0944\t0.0909\n'
'ENSG00000000419.13\tDPM1\tprotein_coding\t2504\t1325\t1179\t142.5398\t41.3097\t39.7597\n'
'ENSG00000000457.14\tSCYL3\tprotein_coding\t1596\t1256\t1248\t15.9318\t4.6172\t4.4440\n'


In [87]:
rna_df.dtypes

gene_id                   str
gene_name                 str
gene_type                 str
unstranded              int64
stranded_first          int64
stranded_second         int64
tpm_unstranded        float64
fpkm_unstranded       float64
fpkm_uq_unstranded    float64
dtype: object

In [88]:
rna_df["gene_type"].value_counts().head(10)

gene_type
protein_coding                        19962
lncRNA                                16901
processed_pseudogene                  10167
unprocessed_pseudogene                 2614
misc_RNA                               2212
snRNA                                  1901
miRNA                                  1881
TEC                                    1057
snoRNA                                  943
transcribed_unprocessed_pseudogene      939
Name: count, dtype: int64

In [89]:
import numpy as np

sample_rna_files = labeled_rna[:10]

print("Number of files:", len(sample_rna_files))

Number of files: 10


In [90]:
for hit in sample_rna_files:
    print(hit["cases"][0]["submitter_id"], hit["file_name"])

TCGA-A8-A09E dadd1789-354a-4abb-8d4e-8556166cf341.rna_seq.augmented_star_gene_counts.tsv
TCGA-A7-A0DC db5dab56-838e-4784-a7cf-f05296e356f7.rna_seq.augmented_star_gene_counts.tsv
TCGA-A7-A0DC c6a90de3-979b-46b0-9f84-2875c1e38742.rna_seq.augmented_star_gene_counts.tsv
TCGA-A7-A0DC 0c814cd9-5b2c-4764-9e6a-6ad0403fcd7d.rna_seq.augmented_star_gene_counts.tsv
TCGA-A8-A08F 616b6b60-8c4a-4d85-9a24-f47df6ad497e.rna_seq.augmented_star_gene_counts.tsv
TCGA-GI-A2C8 03d891b3-8faf-4384-94ce-2015f1ca5df0.rna_seq.augmented_star_gene_counts.tsv
TCGA-E2-A1L7 da752364-731c-4e77-8d6e-9c91a5b7f68f.rna_seq.augmented_star_gene_counts.tsv
TCGA-AO-A0JI a854c368-6e81-4ade-9ea9-afbee1c4c448.rna_seq.augmented_star_gene_counts.tsv
TCGA-A2-A0D4 824f5b0a-44d4-4ccf-9277-1ef752462bb3.rna_seq.augmented_star_gene_counts.tsv
TCGA-AC-A62V 9692b4cc-6c7f-4f88-8854-769568f088ae.rna_seq.augmented_star_gene_counts.tsv


In [95]:
rna_vectors = {}

for hit in sample_rna_files:
    case_id = hit["cases"][0]["submitter_id"]
    file_id = hit["file_id"]

    response = requests.get(
        f"https://api.gdc.cancer.gov/data/{file_id}"
    )
    response.raise_for_status()

    with open("temp_rna.tsv", "wb") as f:
        f.write(response.content)

    df = pd.read_csv(
        "temp_rna.tsv",
        sep="\t",
        skiprows=1
    )

    rna_vectors[case_id] = df.set_index("gene_id")["tpm_unstranded"]

In [96]:
rna_matrix = pd.DataFrame(rna_vectors)

print(rna_matrix.shape)

(60664, 8)


In [97]:
print("Unique genes:", rna_matrix.index.nunique())
print("Total rows:", len(rna_matrix.index))

Unique genes: 60664
Total rows: 60664


In [98]:
print("Missing values:", rna_matrix.isna().sum().sum())

Missing values: 32


In [99]:
print(rna_matrix.head())

                    TCGA-A8-A09E  TCGA-A7-A0DC  TCGA-A8-A08F  TCGA-GI-A2C8  \
gene_id                                                                      
N_unmapped                   NaN           NaN           NaN           NaN   
N_multimapping               NaN           NaN           NaN           NaN   
N_noFeature                  NaN           NaN           NaN           NaN   
N_ambiguous                  NaN           NaN           NaN           NaN   
ENSG00000000003.15       59.4987       15.0719       21.7591        3.2198   

                    TCGA-E2-A1L7  TCGA-AO-A0JI  TCGA-A2-A0D4  TCGA-AC-A62V  
gene_id                                                                     
N_unmapped                   NaN           NaN           NaN           NaN  
N_multimapping               NaN           NaN           NaN           NaN  
N_noFeature                  NaN           NaN           NaN           NaN  
N_ambiguous                  NaN           NaN           NaN        

In [100]:
summary_rows = [
    "N_unmapped",
    "N_multimapping",
    "N_noFeature",
    "N_ambiguous"
]

rna_matrix = rna_matrix.drop(index=summary_rows)

print(rna_matrix.shape)

(60660, 8)


In [101]:
print("Missing values:", rna_matrix.isna().sum().sum())

Missing values: 0


In [102]:
rna_matrix.head()

,TCGA-A8-A09E,TCGA-A7-A0DC,TCGA-A8-A08F,TCGA-GI-A2C8,TCGA-E2-A1L7,TCGA-AO-A0JI,TCGA-A2-A0D4,TCGA-AC-A62V
gene_id,,,,,,,,
ENSG00000000003.15,59.4987,15.0719,21.7591,3.2198,70.8275,54.1708,14.1859,10.6064
ENSG00000000005.6,0.3259,0.8032,0.0527,0.3770,3.6717,0.3428,1.2657,0.6661
ENSG00000000419.13,142.5398,57.8420,158.6434,47.9404,101.8789,136.4973,93.2094,112.2490
ENSG00000000457.14,15.9318,13.8176,15.2103,3.3041,13.4961,12.8766,12.2197,3.3119
ENSG00000000460.17,9.8862,5.4279,6.0498,0.4893,4.4238,10.1232,7.9413,3.8493


In [103]:
gene_nonzero = (rna_matrix > 0).sum(axis=1)

print(gene_nonzero.describe())

count    60660.000000
mean         4.274003
std          3.317862
min          0.000000
25%          1.000000
50%          4.000000
75%          8.000000
max          8.000000
dtype: float64


In [104]:
print("Genes expressed in all patients:",
      (gene_nonzero == rna_matrix.shape[1]).sum())

print("Genes expressed in at least half:",
      (gene_nonzero >= rna_matrix.shape[1] / 2).sum())

print("Genes never expressed:",
      (gene_nonzero == 0).sum())

Genes expressed in all patients: 20803
Genes expressed in at least half: 32739
Genes never expressed: 12980


In [105]:
print(rna_matrix.stack().describe())

count    485280.000000
mean         16.485328
std         428.014599
min           0.000000
25%           0.000000
50%           0.044800
75%           2.067350
max      171277.229300
dtype: float64


In [106]:
print("Maximum TPM:", rna_matrix.max().max())
print("Median TPM:", rna_matrix.stack().median())

Maximum TPM: 171277.2293
Median TPM: 0.0448


RNA expression is highly skewed, so we need to transform it to reduce the dominance of extremely highly expressed genes.

In [107]:
rna_log = np.log1p(rna_matrix)

print(rna_log.stack().describe())

count    485280.000000
mean          0.818683
std           1.359672
min           0.000000
25%           0.000000
50%           0.043825
75%           1.120814
max          12.051045
dtype: float64


In [108]:
print("Maximum after log1p:", rna_log.max().max())
print("Median after log1p:", rna_log.stack().median())

Maximum after log1p: 12.051044585167157
Median after log1p: 0.04382547954002956
